# Preparation

In [1]:
%pip install polars
%pip install altair
%pip install dotenv
%pip install vegafusion
%pip install vl_convert_python

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [97]:
import polars as pl
import os
from dotenv import load_dotenv
from dataclasses import dataclass, field
from typing import Optional
import altair as alt
alt.data_transformers.enable("vegafusion")

DataTransformerRegistry.enable('vegafusion')

In [3]:
DOTENV_PATH = ".env"
load_dotenv(override=True)
os.chdir(os.getenv('PATH_NAME'))
WORK_DIR = os.getcwd()
DATA = os.path.join(WORK_DIR, "data")

In [4]:
files = {
    "quotes_inc_eu": "DE0007500001_quotes_incremental.csv",
    "quotes_inc_us": "US2561631068_quotes_incremental.csv",
    "trades_eu": "DE0007500001_trades.csv",
    "trades_us": "US2561631068_trades.csv"
}

In [5]:
quotes_inc_eu_schema = {
    "trade_id": pl.Int128,
    "event_timestamp": pl.Datetime("ns"),
    "price": pl.Float64,
    "best_bid_price": pl.Float64,
    "best_ask_price": pl.Float64,
}

trades_eu_schema = {
    "trade_id": pl.Int128,
    "event_timestamp": pl.Datetime("ns")
}

quotes_inc_us_schema = {
    "trade_id": pl.Int128,
    "event_timestamp": pl.Datetime("ns")
}

trades_us_schema = {
    "trade_id": pl.Int128,
    "event_timestamp": pl.Datetime("ns")
}

In [6]:
@dataclass
class LimitOrderBook:
    file: str
    folder_path: str
    df: pl.LazyFrame = field(init=False)
    schema_override: Optional[dict] = None
    separator: str = ","

    def __post_init__(self):
        # pl.scan_csv doesn't load the file into memory immediately but only when called with the
        # .collect() method, which generally makes running the code significantly faster.
        # schema_overrides is optional and can be used to explicitly set a data type to a column,
        # but it will return an error if polars finds some kind of mismatch.
        # def get_data(file: str, separator: str = ",", schema_overrides: dict = None) -> pl.LazyFrame:
        self.df = pl.scan_csv(
            source=f"{self.folder_path}/{self.file}",
            separator=self.separator,
            schema_overrides=self.schema_override
        )

@dataclass
class Trades:
    file: str
    folder_path: str
    df: pl.LazyFrame = field(init=False)
    schema_override: Optional[dict] = None
    separator: str = ","
    
    def __post_init__(self):
        self.df = pl.scan_csv(
            source=f"{self.folder_path}/{self.file}",
            separator=self.separator,
            schema_overrides=self.schema_override
        )

In [7]:
## EU: Incremental quotes
quotes_inc_eu = LimitOrderBook(
    file=files["quotes_inc_eu"],
    folder_path=DATA,
    schema_override=quotes_inc_eu_schema
)

quotes_inc_eu = (
    quotes_inc_eu.df.sort(by=[pl.col("original_order_id"), 
                              pl.col("event_timestamp")]
    )
)

trades_eu = Trades(
    file=files["trades_eu"],
    folder_path=DATA,
    schema_override=trades_eu_schema
)

trades_eu = trades_eu.df.sort(by=pl.col("trade_timestamp"))

In [8]:
quotes_inc_eu = quotes_inc_eu. \
    sort(by=[pl.col("original_order_id"), pl.col("event_timestamp")]) \
    .with_columns(
        pl.when(pl.col("trade_id") == 0) \
            .then(None) \
            .otherwise(pl.col("trade_id")) \
            .alias("trade_id"),
        pl.col("size") \
            .diff() \
            .over("original_order_id")
            .fill_null(pl.col("size"))
            .alias("size_diff"),
        pl.col("event_timestamp") \
            .rank(method="ordinal") \
            .over("original_order_id") \
            .alias("order_order"),
        pl.col("size") \
            .diff() \
            .over("original_order_id") \
            .fill_null(0) \
            .cum_sum() \
            .over("original_order_id")
            .alias("size_diff_cum_sum")
    )

In [9]:
# Sanity check: No original_order_id's exist within more than one venue.
print(
    quotes_inc_eu.group_by("original_order_id") \
        .agg([
            pl.col("venue").unique().alias("venues")
        ])
        .filter(pl.col("venues").list.len() > 1)
        .collect()
)

shape: (0, 2)
┌───────────────────┬───────────┐
│ original_order_id ┆ venues    │
│ ---               ┆ ---       │
│ i64               ┆ list[str] │
╞═══════════════════╪═══════════╡
└───────────────────┴───────────┘


In [10]:
def show_histogram(df: pl.LazyFrame, column_name: str, x_label: str = None, y_label: str = None, sort_order: list | str = None) -> None:
    
    x=alt.X(
        f"{column_name}:N", 
        sort=alt.SortField(sort_order if sort_order is not None else alt.Undefined), 
        axis=alt.Axis(labelAngle=0, title=x_label or column_name)
    )
    
    (
        alt.layer(
            alt.Chart(df.collect()).mark_bar().encode(
                x=x,
                y="count():Q",
            ),
            alt.Chart(df.collect()).mark_text(dy=-5).encode(
                x=x,
                y="count():Q",
                text="count():Q",
            ),
        )
        .properties(width=750, height=250)
    ).display()

In [11]:
show_histogram(
    df=quotes_inc_eu, 
    column_name="venue",
    x_label="Venues",
    sort_order="venue"
)
show_histogram(
    df=quotes_inc_eu, 
    column_name="market_state",
    x_label="Market states",
    sort_order="market_state"
)
show_histogram(
    df=quotes_inc_eu, 
    column_name="lob_action",
    x_label="LOB Actions",
    sort_order="lob_action"
)

alt.LayerChart(...)

alt.LayerChart(...)

alt.LayerChart(...)

In [12]:
# Sanity check: Every order should have a REMOVE operation at some point
# and at some market state
orders_without_remove = (
    quotes_inc_eu.group_by(pl.col("original_order_id"))
    .agg(pl.col("lob_action").unique().alias("lob_actions"))
    .filter(
        # pl.col("lob_actions").list.contains("INSERT") &
        pl.col("lob_actions").list.contains("REMOVE").not_()
    )
)

orders_without_remove.collect().show(limit=20)

original_order_id,lob_actions
i64,list[str]


In [13]:
# Ideally, all trade_id's in the LOB would be found in the trades dataset,
# but this does not seem to be the case:
# I take all unique trade_id's in the LOB and I compare them against the trades dataset.
# Interestingly, the relevant trade_ids are associated only aggressor_side = UNKNOWN.
# Changing to from anti to inner join in trade_id_check will return BID and ASK only.
trade_ids_unique = quotes_inc_eu.select(pl.col("trade_id")).unique()

trade_id_check = trades_eu.join(
    trade_ids_unique,
    on="trade_id",
    how="anti"
)

show_histogram(
    df=trade_id_check, 
    column_name="trade_type",
    x_label="Trade Types",
    sort_order="trade_type"
)

show_histogram(
    df=trade_id_check, 
    column_name="aggressor_side",
    x_label="Aggressor Side",
    sort_order="aggressor_side"
)

trade_id_check.collect().sample(10)

alt.LayerChart(...)

alt.LayerChart(...)

trade_id,trade_timestamp,publication_timestamp,aggressor_side,price,execution_size,market_state,trade_type,venue
i128,str,str,str,f64,i64,str,str,str
42943561947635,"""2023-09-01 07:36:55.202383000""","""2023-09-01 07:36:55.202383000""","""UNKNOWN""",7.204,411,"""CONTINUOUS_TRADING""","""DARK""","""CEUX"""
42943562000313,"""2023-09-01 10:40:01.370121000""","""2023-09-01 10:40:01.370121000""","""UNKNOWN""",7.292,576,"""CONTINUOUS_TRADING""","""DARK""","""CEUX"""
42943562009866,"""2023-09-01 11:27:31.577145000""","""2023-09-01 11:27:31.577145000""","""UNKNOWN""",7.286,660,"""CONTINUOUS_TRADING""","""DARK""","""CEUX"""
42943562113144,"""2023-09-01 15:26:47.300650000""","""2023-09-01 15:26:47.300650000""","""UNKNOWN""",7.372,558,"""CONTINUOUS_TRADING""","""DARK""","""CEUX"""
211963,"""2023-09-01 15:35:07.465751000""","""2023-09-01 15:35:07.465751000""","""UNKNOWN""",7.37,619,"""POST_TRADE""","""CLOSING_PRICE""","""AQEU"""
160382,"""2023-09-01 14:23:58.419691000""","""2023-09-01 14:23:58.419691000""","""UNKNOWN""",7.395,750,"""CONTINUOUS_TRADING""","""DARK""","""AQEU"""
42943561955435,"""2023-09-01 07:59:53.944019000""","""2023-09-01 07:59:53.944019000""","""UNKNOWN""",7.272,635,"""CONTINUOUS_TRADING""","""DARK""","""CEUX"""
42943562011897,"""2023-09-01 11:36:30.462660000""","""2023-09-01 11:36:30.462660000""","""UNKNOWN""",7.305,659,"""CONTINUOUS_TRADING""","""DARK""","""CEUX"""
6269133319539,"""2023-09-01 15:29:40.832500000""","""2023-09-01 15:29:40.832500000""","""UNKNOWN""",7.366,20,"""AUCTION_ON_DEMAND""","""UNCROSSING""","""CEUX"""


# 3.1.

In [14]:
# Here I'm preparing the one-row-per-order dataset from the LOB.
# quotes_aggs 
quotes_aggs = [
    pl.col("venue") \
        .first()
        .alias("venue"),
    pl.col("event_timestamp") \
        .filter(pl.col("lob_action")=="INSERT")
        .first()
        .alias("insertion_date"),
    pl.col("event_timestamp") \
        .filter(pl.col("lob_action")=="REMOVE")
        .first()
        .alias("removal_date"),
    pl.col("event_timestamp") \
        .max()
        .alias("latest_event_timestamp"),
    pl.col("lob_action") \
        .eq("UPDATE")
        .sum()
        .alias("number_of_updates"),
    pl.col("price") \
        .filter(pl.col("lob_action")=="INSERT")
        .first()
        .alias("price_at_insertion"),
    pl.col("size") \
        .filter(pl.col("lob_action")=="INSERT")
        .first()
        .alias("size_at_insertion"),
    pl.col("old_price") \
        .filter(pl.col("lob_action")=="REMOVE")
        .first()
        .alias("old_price_at_removal"),
    pl.col("old_size") \
        .filter(pl.col("lob_action")=="REMOVE")
        .first()
        .alias("old_size_at_removal"),
    # EXECUTION SIZE
    # This sums the execution sizes of all original_order_id's
    # in preparation for determining the removal mechanism.
    pl.col("execution_size") \
        .filter(pl.col("order_executed")==True)
        .sum() # .first()
        .alias("execution_size"),
    pl.col("price_level") \
        .filter(pl.col("lob_action")=="INSERT")
        .first()
        .alias("insertion_level"),
    pl.col("best_bid_price") \
        .filter(pl.col("lob_action")=="INSERT")
        .fill_null(0)
        .first()
        .alias("best_bid_price_at_insertion"),
    pl.col("best_ask_price") \
        .filter(pl.col("lob_action")=="INSERT")
        .first()
        .alias("best_ask_price_at_insertion"),
    pl.struct(
        [
            pl.col("event_timestamp") \
                .filter(pl.col("lob_action")=="UPDATE")
                .alias("lob_updates"),
            pl.col("price") \
                .filter(pl.col("lob_action")=="UPDATE")
                .alias("updated_prices"),
            pl.col("size") \
                .filter(pl.col("lob_action")=="UPDATE")
                .alias("updated_sizes"),
        ]
    ),
    pl.struct(
        [
            pl.col("trade_id") \
                .filter((pl.col("trade_id").is_not_null()) | 
                        (pl.col("trade_id") != 0)) \
                .unique() \
                .alias("associated_trade_ids")
        ]
    ),
    pl.col("market_state") \
        .filter(pl.col("lob_action")=="REMOVE")
        .first()
        .alias("market_state_at_removal"),
    pl.col("size_diff") \
        .filter((pl.col("lob_action")!="REMOVE") & 
                (pl.col("order_executed")==False)) \
        .sum()
        .alias("size_diff_tally"),
]

quotes_calc = [
    (
        pl.when(pl.col("removal_date").is_not_null())
            .then(pl.col("removal_date"))
            .otherwise(pl.col("latest_event_timestamp"))
        - pl.col("insertion_date")
    ).dt.total_microseconds().alias("order_lifetime"),

    # (best bid price + best ask price) / 2
    (
        (
            pl.col("best_bid_price_at_insertion")
            + pl.col("best_ask_price_at_insertion")
        ) / 2
    ).alias("midpoint_at_insertion"),
]

In [15]:
quotes_collapsed = (
    quotes_inc_eu
    .group_by("original_order_id")
    .agg(quotes_aggs)
    .with_columns(quotes_calc)
)

log_order_lifetime_bins = [2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 30]
log_order_lifetime_bin_labels = ['00-02', '02-04', '04-06', '06-08', '08-10', '10-12', '12-14', '14-16', '16-18', '18-20', '20-22', '22-24', '24-30', '>=30']

quotes_collapsed = quotes_collapsed.with_columns([
    # Absolute distance to midpoint:
    # |Price at insertion - Midpoint at Insertion|
    (
        pl.col("price_at_insertion")
        .sub(pl.col("midpoint_at_insertion"))
        .abs()
        .alias("abs_distance_to_midpoint")
    ),
    # Relative distance to midpoint:
    # |Price at insertion - Midpoint at Insertion|
    # / Midpoint at Insertion
    (
        (
            pl.col("price_at_insertion")
            .sub(pl.col("midpoint_at_insertion"))
            .abs()
        )
        .truediv(pl.col("midpoint_at_insertion"))
        .alias("rel_distance_to_midpoint")
    ),
    # TODO: Converted to microseconds
    # Log order lifetime
    # All order lifetimes need to be > 0, otherwise this would throw an error.
    # Further note: The durations in order_lifetime are implicitly casted
    # as Unix epoch nanoseconds and then the logarithm is calculated.
    # In quotes_calc, I've converted order_lifetime to microseconds but, since
    # the shortest order lifetime (at least in the EU dataset) is 2 microseconds,
    # which is 2000ns, I am de facto not sacrificing any accuracy.
    # At the same time, I'm also repeating the the same logarith but with bins
    # to prepare for 3.2.
    (
        pl.col("order_lifetime")
            .log1p()
            .alias("log_order_lifetime")
    ),
    (
        pl.col("order_lifetime")
            .log1p()
            .cut(
                breaks= log_order_lifetime_bins,
                labels=log_order_lifetime_bin_labels
            )
            .cast(pl.Utf8)
            .alias("log_order_lifetime_bins")          
    ),
    # Removal mechanism:
    # The execution size here is the final (cumulative) execution size.
    # Furthermore, we need to take into account the cases, in which an order has had its size modified (regardless of any associated trades).
    # over its lifecycle. I do this by:
    # 1. Adding all (positive) size_diff's for non-executed orders, but including the INSERT order and excluding the REMOVE order,
    # 2. Determining the difference between the size tally from 1. and the final execution size.
    # Thus, if
    #   execution_size = 0 (or size tally - execution_size = size tally), then none of the order has been filled and it has been cancelled,
    #   size tally - execution_size = 0 (or size tally = execution_size), then the entire order has been filled and then removed,
    #   size tally - execution_size > 0 (or size tally > execution_size), then the order has been partially filled, but still cancelled.
    (
       pl.when(pl.col("execution_size") == 0) \
            .then(pl.lit("Cancel")) \
            .when(pl.col("execution_size")==pl.col("size_diff_tally")) \
            .then(pl.lit("Trade (full)")) \
            .when(pl.col("execution_size").is_between(0, pl.col("size_diff_tally"), closed="none")) \
            .then(pl.lit("Trade (partial)")) \
            .otherwise(pl.lit("Unknown"))
        .alias("removal_mechanism")
    ),
])

In [16]:
# Sanity check: No order should have an order lifetime <= 0,
# which would mess up the logarithm.
quotes_collapsed \
    .filter(pl.col("order_lifetime") <= 0) \
    .collect() \
    .show(limit=10)

original_order_id,venue,insertion_date,removal_date,latest_event_timestamp,number_of_updates,price_at_insertion,size_at_insertion,old_price_at_removal,old_size_at_removal,execution_size,insertion_level,best_bid_price_at_insertion,best_ask_price_at_insertion,lob_updates,associated_trade_ids,market_state_at_removal,size_diff_tally,order_lifetime,midpoint_at_insertion,abs_distance_to_midpoint,rel_distance_to_midpoint,log_order_lifetime,log_order_lifetime_bins,removal_mechanism
i64,str,datetime[ns],datetime[ns],datetime[ns],u32,f64,i64,f64,i64,i64,i64,f64,f64,list[struct[3]],list[struct[1]],str,i64,i64,f64,f64,f64,f64,str,str


In [17]:
# Sanity check: No order's removal mechanism should be classified as "Unknown"
quotes_collapsed \
    .filter(pl.col("removal_mechanism")=="Unknown") \
    .collect() \
    .show(limit=10)

original_order_id,venue,insertion_date,removal_date,latest_event_timestamp,number_of_updates,price_at_insertion,size_at_insertion,old_price_at_removal,old_size_at_removal,execution_size,insertion_level,best_bid_price_at_insertion,best_ask_price_at_insertion,lob_updates,associated_trade_ids,market_state_at_removal,size_diff_tally,order_lifetime,midpoint_at_insertion,abs_distance_to_midpoint,rel_distance_to_midpoint,log_order_lifetime,log_order_lifetime_bins,removal_mechanism
i64,str,datetime[ns],datetime[ns],datetime[ns],u32,f64,i64,f64,i64,i64,i64,f64,f64,list[struct[3]],list[struct[1]],str,i64,i64,f64,f64,f64,f64,str,str


In [18]:
# Result
quotes_collapsed.collect().show(limit=10)
quotes_collapsed.collect().describe()

original_order_id,venue,insertion_date,removal_date,latest_event_timestamp,number_of_updates,price_at_insertion,size_at_insertion,old_price_at_removal,old_size_at_removal,execution_size,insertion_level,best_bid_price_at_insertion,best_ask_price_at_insertion,lob_updates,associated_trade_ids,market_state_at_removal,size_diff_tally,order_lifetime,midpoint_at_insertion,abs_distance_to_midpoint,rel_distance_to_midpoint,log_order_lifetime,log_order_lifetime_bins,removal_mechanism
i64,str,datetime[ns],datetime[ns],datetime[ns],u32,f64,i64,f64,i64,i64,i64,f64,f64,list[struct[3]],list[struct[1]],str,i64,i64,f64,f64,f64,f64,str,str
50137,"""AQEU""",2023-09-01 07:00:23.832485,2023-09-01 11:00:00.100847,2023-09-01 11:00:00.100847,0,8.1,947,8.1,947,0,1,0.0,8.1,[],[],"""CONTINUOUS_TRADING""",947,14376268362,4.05,4.05,1.0,23.388845,"""22-24""","""Cancel"""
50138,"""AQEU""",2023-09-01 07:00:23.832985,2023-09-01 11:00:00.100871,2023-09-01 11:00:00.100871,0,5.0,757,5.0,757,0,1,5.0,8.1,[],[],"""CONTINUOUS_TRADING""",757,14376267886,6.55,1.55,0.236641,23.388845,"""22-24""","""Cancel"""
51441,"""AQEU""",2023-09-01 07:00:24.633269,2023-09-01 07:00:24.642583,2023-09-01 07:00:24.642583,0,7.104,750,7.104,750,0,1,7.104,8.1,[],[],"""CONTINUOUS_TRADING""",750,9314,7.602,0.498,0.065509,9.139381,"""08-10""","""Cancel"""
51442,"""AQEU""",2023-09-01 07:00:24.633271,2023-09-01 07:00:24.633286,2023-09-01 07:00:24.633286,0,7.102,750,7.102,750,0,2,7.104,8.1,[],[],"""CONTINUOUS_TRADING""",750,15,7.602,0.5,0.065772,2.772589,"""02-04""","""Cancel"""
51443,"""AQEU""",2023-09-01 07:00:24.633283,2023-09-01 07:00:24.642585,2023-09-01 07:00:24.642585,0,7.104,750,7.104,750,0,1,7.104,8.1,[],[],"""CONTINUOUS_TRADING""",750,9302,7.602,0.498,0.065509,9.138092,"""08-10""","""Cancel"""
60099,"""AQEU""",2023-09-01 07:00:30.221642,2023-09-01 07:00:30.228353,2023-09-01 07:00:30.228353,0,7.112,750,7.112,750,0,1,7.112,8.1,[],[],"""CONTINUOUS_TRADING""",750,6711,7.606,0.494,0.064949,8.811652,"""08-10""","""Cancel"""
60135,"""AQEU""",2023-09-01 07:00:30.228330,2023-09-01 07:00:30.328332,2023-09-01 07:00:30.328332,0,7.114,1,7.114,1,0,1,7.114,8.1,[],[],"""CONTINUOUS_TRADING""",1,100002,7.607,0.493,0.064809,11.512955,"""10-12""","""Cancel"""
60143,"""AQEU""",2023-09-01 07:00:30.230594,2023-09-01 07:00:30.376114,2023-09-01 07:00:30.376114,0,7.116,750,7.116,750,0,1,7.116,8.1,[],[],"""CONTINUOUS_TRADING""",750,145520,7.608,0.492,0.064669,11.888076,"""10-12""","""Cancel"""
66381,"""AQEU""",2023-09-01 07:00:35.319647,2023-09-01 07:32:24.281250,2023-09-01 07:32:24.281250,0,7.18,203,7.18,203,0,1,5.0,7.18,[],[],"""CONTINUOUS_TRADING""",203,1908961603,6.09,1.09,0.178982,21.369825,"""20-22""","""Cancel"""


statistic,original_order_id,venue,insertion_date,removal_date,latest_event_timestamp,number_of_updates,price_at_insertion,size_at_insertion,old_price_at_removal,old_size_at_removal,execution_size,insertion_level,best_bid_price_at_insertion,best_ask_price_at_insertion,lob_updates,associated_trade_ids,market_state_at_removal,size_diff_tally,order_lifetime,midpoint_at_insertion,abs_distance_to_midpoint,rel_distance_to_midpoint,log_order_lifetime,log_order_lifetime_bins,removal_mechanism
str,f64,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,str,str
"""count""",189449.0,"""189449""","""189449""","""189449""","""189449""",189449.0,189449.0,189449.0,189449.0,189449.0,189449.0,189449.0,189449.0,189438.0,189449.0,189449.0,"""189449""",189449.0,189449.0,189438.0,189438.0,189438.0,189449.0,"""189449""","""189449"""
"""null_count""",0.0,"""0""","""0""","""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11.0,0.0,0.0,"""0""",0.0,0.0,11.0,11.0,11.0,0.0,"""0""","""0"""
"""mean""",1.1159e18,null,"""2023-09-01 10:53:25.881948""","""2023-09-01 10:55:34.293706""","""2023-09-01 10:56:33.331417""",0.274121,7.312965,610.24908,7.313094,594.239722,23.358566,5.146852,7.298362,7.312898,null,null,null,607.826951,1.2841e8,7.305635,0.031021,0.004357,14.193243,null,null
"""std""",6.7317e17,null,null,null,null,7.522872,0.393784,989.119931,0.393746,950.20554,316.824017,10.091634,0.144286,0.085213,null,null,null,1176.221348,1.0115e9,0.105524,0.37751,0.053253,3.878808,null,null
"""min""",50137.0,"""AQEU""","""2023-09-01 05:30:04.652687""","""2023-09-01 07:00:12.541504""","""2023-09-01 07:00:12.541504""",0.0,0.001,1.0,0.001,1.0,0.0,1.0,0.0,7.086,null,null,"""CLOSED""",1.0,2.0,4.05,0.001,0.000134,1.098612,"""00-02""","""Cancel"""
"""25%""",2.1376e17,null,"""2023-09-01 08:14:15.609704""","""2023-09-01 08:17:57.923187""","""2023-09-01 08:17:57.923187""",0.0,7.27,259.0,7.27,250.0,0.0,1.0,7.27,7.28,null,null,null,250.0,134419.0,7.275,0.003,0.000422,11.808725,null,null
"""50%""",1.0814e18,null,"""2023-09-01 10:40:01.397597""","""2023-09-01 10:46:27.364529""","""2023-09-01 10:46:27.364529""",0.0,7.318,379.0,7.318,378.0,0.0,2.0,7.312,7.32,null,null,null,378.0,3.341794e6,7.316,0.007,0.000941,15.022019,null,null
"""75%""",1.6936e18,null,"""2023-09-01 13:33:46.024652""","""2023-09-01 13:35:27.259075""","""2023-09-01 13:35:48.813363""",0.0,7.382,637.0,7.382,635.0,0.0,5.0,7.38,7.386,null,null,null,637.0,2.2385409e7,7.383,0.019,0.00267,16.92392,null,null
"""max""",1.6936e18,"""XMIL""","""2023-09-01 15:30:00.610934""","""2023-09-01 15:40:00.008912""","""2023-09-01 15:40:00.008912""",1522.0,70.0,80000.0,70.0,80000.0,35004.0,247.0,7.44,8.1,null,null,"""POST_TRADE""",160000.0,3.6595e10,7.692,62.883,8.835605,24.323187,"""24-30""","""Trade (partial)"""


# 3.2

In [19]:
# This also includes the orders that are not found in the LOB
alt.Chart(trades_eu.collect()).mark_bar().encode(
    x = alt.X('venue:N', axis=alt.Axis(labelAngle=0), title="Venues"),
    xOffset = 'trade_type:N',
    y=alt.Y('count():Q', title="No. of Trades"),
    color=alt.Color('trade_type:N', title="Trade type"),
    tooltip=[
        alt.Tooltip("venue:N", title="Venue"),
        alt.Tooltip("count():Q", title="No. of Trades"),
    ],
)

alt.Chart(...)

In [20]:
quotes_collapsed \
    .group_by("venue") \
    .agg(
        [
            pl.col("venue")
                .count()
                .alias("total_number_of_orders"),
            pl.col("number_of_updates")
                .sum()
                .alias("total_number_of_updates"),
            pl.col("number_of_updates")
                .mean()
                .alias("avg_number_of_updates_per_order"),
            pl.col("order_lifetime")
                .mean()
                .cast(pl.Duration)
                .alias("avg_order_lifetime"),
            pl.col("order_lifetime")
                .median()
                .cast(pl.Duration)
                .alias("median_order_lifetime"),
            pl.col("execution_size")
                .sum()
                .alias("total_executed_size"),
            pl.col("removal_mechanism")
                .filter(pl.col("removal_mechanism")=="Cancel")
                .count()
                .truediv(pl.col("venue").count())
                .alias("rel_number_of_cancels"),
            pl.col("removal_mechanism")
                .filter(pl.col("removal_mechanism")=="Trade (full)").count()
                .truediv(pl.col("venue").count())
                .alias("rel_number_of_full_trades"),
            pl.col("removal_mechanism")
                .filter(pl.col("removal_mechanism")=="Trade (partial)").count()
                .truediv(pl.col("venue").count())
                .alias("rel_number_of_partial_trades"),
            pl.col("rel_distance_to_midpoint")
                .mean()
                .alias("avg_relative_distance_to_midpoint"),
            pl.col("rel_distance_to_midpoint")
                .median()
                .alias("median_relative_distance_to_midpoint"),
        ]
    ) \
    .sort(by="venue") \
    .collect()

venue,total_number_of_orders,total_number_of_updates,avg_number_of_updates_per_order,avg_order_lifetime,median_order_lifetime,total_executed_size,rel_number_of_cancels,rel_number_of_full_trades,rel_number_of_partial_trades,avg_relative_distance_to_midpoint,median_relative_distance_to_midpoint
str,u32,u32,f64,duration[μs],duration[μs],i64,f64,f64,f64,f64,f64
"""AQEU""",37184,3099,0.083342,17s 137378µs,405132µs,98851,0.991502,0.006912,0.001587,0.0043,0.000823
"""CEUX""",49581,11113,0.224138,1m 4s 440993µs,5s 118348µs,739598,0.952744,0.04421,0.003046,0.00396,0.001893
"""TQEX""",10884,5642,0.518376,47s 896102µs,1s 826191µs,94742,0.972253,0.024531,0.003216,0.004595,0.001496
"""XETR""",85575,15930,0.186152,3m 43s 408205µs,5s 363332µs,3453737,0.951072,0.044663,0.004265,0.004667,0.000687
"""XMIL""",6225,16148,2.594056,2m 17s 467553µs,6s 525311µs,38329,0.994538,0.004016,0.001446,0.003184,0.002674


There is considerable right skewness in the order lifetimes across all venues, meaning most orders live briefly, but some very persistent
orders inflate the average. Similarly, though not as pronounced, this applies to the rel. distance to the midpoint.

#### AQEU
- Relatively close to the midpoint for most orders, but some traders distort the average by placing (passive) orders far away in the book
- The shortest average (and median) order lifetimes by far, most orders disappear almost immediately (cancelled)
- Could be indicative of HFT & quote stuffing


#### CEUX
- Moderate order lifetimes overall, moderate update frequence, moderate rel. distance to midpoint on median (but the opposite on avg.), moderate-to-high execution share
- High rate of trades that are filled fully.
- CEUX seems to have the most dark trades over all venues by far. Generally, this tends to fragment price discovery. Not unheard of in an MTF.


#### TQEX
- Most orders go through very fast and most are not inserted as close as they can be to the midpoint.
- Releatively low update frequency
- Doesn't seem that aggressive of a market


#### XETR
- Highest execution share by far
- Most orders are very close to the midpoint but, also, some quote very far from the midpoint and exist for relatively long time, could be large (passive) institutional orders
- It's Xetra, it's a regulated market


#### XMIL
- Quite high update frequency at very high cancellation rates. Could indicate active management.
- Order are generally quoted relatively far from the midpoint and they have relatively long lifetimes.

In [21]:
# Ideally, the interactive tooltip will work when hovered over, which will show
# relevant information if a bar is not easy to see.
alt.Chart(quotes_collapsed.collect(), title="Log order life times for entire LOB per venue").mark_bar().encode(
    x=alt.X(
        "log_order_lifetime_bins:N",
        axis=alt.Axis(labelAngle=0),
        title="Log order lifetimes",
    ),
    xOffset="venue:N",
    y=alt.Y(
        "count():Q",
        title="No. of Trades",
    ),
    color=alt.Color(
        "venue:N",
        title="Venue",
    ),
    tooltip=[
        alt.Tooltip("log_order_lifetime_bins:N", title="Log order lifetime"),
        alt.Tooltip("venue:N", title="Venue"),
        alt.Tooltip("count():Q", title="No. of Trades"),
    ],
)

alt.Chart(...)

# 3.3

### First idea:
Use hard, non-probabilistic rules to classify trader types. Having high-precision filters makes it easy <br>
to get *a* dataset with clear-cut true's & false's but, at least for the European dataset, around half <br>
of the orders are classified as false across all four types. There are several conceivable reasons for <br>
why this could be, noise, correlated variables (i.e. lifetime, updates, size) etc. The effect is <br>
essentially multiplicative, which is another significant constraint. Switching from AND's to OR's <br>
is also not really an option because it's difficult to account for how or if the individual rules are met.

In [118]:
# Note (2026-05-11: modifying insertion_level from 2 to 3 does nothing, going from 2 to 1 has a slight change
hft_filter = (
    (pl.col("log_order_lifetime")
     <= pl.col("log_order_lifetime").quantile(0.25, 'nearest'))
    & (pl.col("size_at_insertion") 
       <= pl.col("size_at_insertion").quantile(0.5, 'nearest'))
    & (pl.col("number_of_updates") 
       >= pl.col("number_of_updates").quantile(0.75, 'nearest'))
    & (pl.col("rel_distance_to_midpoint") 
       <= pl.col("rel_distance_to_midpoint").quantile(0.25, 'nearest'))
    & (pl.col("insertion_level") <= 3)
)

market_maker_filter = (
    (pl.col("insertion_level") == 1)
    & (pl.col("number_of_updates") 
       >= pl.col("number_of_updates").quantile(0.5, 'nearest'))
    & (pl.col("log_order_lifetime")
        .is_between(pl.col("log_order_lifetime").quantile(0.25, 'nearest')
                    , pl.col("log_order_lifetime").quantile(0.75, 'nearest')
                    , closed='both'))
    & (pl.col("size_at_insertion")
        .is_between(pl.col("size_at_insertion").quantile(0.25, 'nearest')
                    , pl.col("size_at_insertion").quantile(0.75, 'nearest')
                    , closed='both'))
)

institutional_filter = (
    (pl.col("size_at_insertion") 
     >= pl.col("size_at_insertion").quantile(0.8, 'nearest'))
    & (pl.col("log_order_lifetime") 
       >= pl.col("log_order_lifetime").quantile(0.75, 'nearest'))
    & (pl.col("number_of_updates") 
       <= pl.col("number_of_updates").quantile(0.25, 'nearest'))
)

retail_filter = (
    (pl.col("size_at_insertion") 
     <= pl.col("size_at_insertion").quantile(0.5, 'nearest'))
    & (pl.col("number_of_updates") 
       <= pl.col("number_of_updates").quantile(0.5, 'nearest'))
    & (pl.col("log_order_lifetime")
        .is_between(pl.col("log_order_lifetime").quantile(0.2, 'nearest')
                    , pl.col("log_order_lifetime").quantile(0.8, 'nearest')))
)

In [ ]:
def heuristics_by_venue(data: pl.LazyFrame
                        , venue: str | list[str] = None) -> pl.LazyFrame:
    if isinstance(venue, str):
        venue = [venue]

    df = data \
            .filter(pl.col("venue").is_in(venue) if venue is not None else True) \
            .with_columns(
                pl.when(hft_filter)
                    .then(True)
                    .otherwise(False)
                    .alias("is_hft"),
                pl.when(market_maker_filter)
                    .then(True)
                    .otherwise(False)
                    .alias("is_market_maker"),
                pl.when(institutional_filter)
                    .then(True)
                    .otherwise(False)
                    .alias("is_institutional"),
                pl.when(retail_filter)
                    .then(True)
                    .otherwise(False)
                    .alias("is_retail")
            )
    
    return df

In [114]:
heuristic_xetr = heuristics_by_venue(quotes_collapsed, "XETR")
heuristic_xetr \
    .filter(
        (pl.col("is_hft")==False)
        & (pl.col("is_market_maker")==False)
        & (pl.col("is_institutional")==False)
        & (pl.col("is_retail")==False)       
    ) \
    .collect()

original_order_id,venue,insertion_date,removal_date,latest_event_timestamp,number_of_updates,price_at_insertion,size_at_insertion,old_price_at_removal,old_size_at_removal,execution_size,insertion_level,best_bid_price_at_insertion,best_ask_price_at_insertion,lob_updates,associated_trade_ids,market_state_at_removal,size_diff_tally,order_lifetime,midpoint_at_insertion,abs_distance_to_midpoint,rel_distance_to_midpoint,log_order_lifetime,log_order_lifetime_bins,removal_mechanism,is_hft,is_market_maker,is_institutional,is_retail
i64,str,datetime[ns],datetime[ns],datetime[ns],u32,f64,i64,f64,i64,i64,i64,f64,f64,list[struct[3]],list[struct[1]],str,i64,i64,f64,f64,f64,f64,str,str,bool,bool,bool,bool
1675665317597835782,"""XETR""",2023-09-01 07:00:23.819689604,2023-09-01 11:00:00.096401017,2023-09-01 15:30:00.653300330,0,8.5,290,8.5,290,0,125,7.114,7.12,[],[],"""INTRADAY_AUCTION""",580,14376276711,7.117,1.383,0.194323,23.388845,"""22-24""","""Cancel""",false,false,false,false
1675665332046256478,"""XETR""",2023-09-01 07:00:23.819743366,2023-09-01 11:00:00.096401017,2023-09-01 15:30:00.653300330,0,11.0,236,11.0,236,0,184,7.114,7.12,[],[],"""INTRADAY_AUCTION""",472,14376276657,7.117,3.883,0.545595,23.388845,"""22-24""","""Cancel""",false,false,false,false
1675665901698363101,"""XETR""",2023-09-01 07:00:23.819736033,2023-09-01 11:00:00.096401017,2023-09-01 15:30:00.653300330,0,10.5,210,10.5,210,0,176,7.114,7.12,[],[],"""INTRADAY_AUCTION""",420,14376276664,7.117,3.383,0.475341,23.388845,"""22-24""","""Cancel""",false,false,false,false
1675665904897989796,"""XETR""",2023-09-01 07:00:23.819743366,2023-09-01 11:00:00.096401017,2023-09-01 15:30:00.653300330,0,11.75,150,11.75,150,0,194,7.114,7.12,[],[],"""INTRADAY_AUCTION""",300,14376276657,7.117,4.633,0.650977,23.388845,"""22-24""","""Cancel""",false,false,false,false
1675667360713589796,"""XETR""",2023-09-01 07:00:23.819726522,2023-09-01 11:00:00.096401017,2023-09-01 15:30:00.653300330,0,10.0,100,10.0,100,0,170,7.114,7.12,[],[],"""INTRADAY_AUCTION""",200,14376276674,7.117,2.883,0.405086,23.388845,"""22-24""","""Cancel""",false,false,false,false
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1693582200053431727,"""XETR""",2023-09-01 15:30:00.053442144,2023-09-01 15:30:00.653300330,2023-09-01 15:30:00.653300330,0,6.832,650,6.832,650,0,56,7.36,7.366,[],[],"""CLOSING_AUCTION""",650,599858,7.363,0.531,0.072117,13.30445,"""12-14""","""Cancel""",false,false,false,false
1693582200061963832,"""XETR""",2023-09-01 15:30:00.061973363,2023-09-01 15:30:00.653300330,2023-09-01 15:30:00.653300330,0,7.384,1000,7.384,1000,0,10,7.36,7.366,[],[],"""CLOSING_AUCTION""",1000,591326,7.363,0.021,0.002852,13.290124,"""12-14""","""Cancel""",false,false,false,false
1693582200069257055,"""XETR""",2023-09-01 15:30:00.069273032,2023-09-01 15:30:00.653300330,2023-09-01 15:30:00.653300330,0,6.9,1200,6.9,1200,0,53,7.36,7.366,[],[],"""CLOSING_AUCTION""",1200,584027,7.363,0.463,0.062882,13.277704,"""12-14""","""Cancel""",false,false,false,false


### Second idea

In [120]:
hft_score = (
    1 * (pl.col("log_order_lifetime")
         <= pl.col("log_order_lifetime").quantile(0.25, 'nearest'))
    + 1 * (pl.col("size_at_insertion") 
           <= pl.col("size_at_insertion").quantile(0.5, 'nearest'))
    + 1 * (pl.col("number_of_updates") 
           >= pl.col("number_of_updates").quantile(0.75, 'nearest'))
    + 1 * (pl.col("rel_distance_to_midpoint") 
           <= pl.col("rel_distance_to_midpoint").quantile(0.25, 'nearest'))
    + 1 * (pl.col("insertion_level") <= 3)
)

market_maker_score = (
    1 * (pl.col("insertion_level") == 1)
    + 1 * (pl.col("number_of_updates") 
       >= pl.col("number_of_updates").quantile(0.5, 'nearest'))
    + 1 * (pl.col("log_order_lifetime")
        .is_between(pl.col("log_order_lifetime").quantile(0.25, 'nearest')
                    , pl.col("log_order_lifetime").quantile(0.75, 'nearest')
                    , closed='both'))
    + 1 * (pl.col("size_at_insertion")
        .is_between(pl.col("size_at_insertion").quantile(0.25, 'nearest')
                    , pl.col("size_at_insertion").quantile(0.75, 'nearest')
                    , closed='both'))
)

institutional_score = (
    1 * (pl.col("size_at_insertion") 
     >= pl.col("size_at_insertion").quantile(0.8, 'nearest'))
    + 1 * (pl.col("log_order_lifetime") 
       >= pl.col("log_order_lifetime").quantile(0.75, 'nearest'))
    + 1 * (pl.col("number_of_updates") 
       <= pl.col("number_of_updates").quantile(0.25, 'nearest'))
)

retail_score = (
    1 * (pl.col("size_at_insertion") 
     <= pl.col("size_at_insertion").quantile(0.5, 'nearest'))
    + 1 * (pl.col("number_of_updates") 
       <= pl.col("number_of_updates").quantile(0.5, 'nearest'))
    + 1 * (pl.col("log_order_lifetime")
        .is_between(pl.col("log_order_lifetime").quantile(0.2, 'nearest')
                    , pl.col("log_order_lifetime").quantile(0.8, 'nearest')))
)

In [127]:
heuristic_alt = (
    quotes_collapsed
        .filter(pl.col("venue")=="XETR")
        .with_columns(
            (hft_score).alias("hft_score")
            , (market_maker_score).alias("market_maker_score")
            , (institutional_score).alias("institutional_score")
            , (retail_score).alias("retail_score")
        )
)
heuristic_alt.filter(
    (pl.col("hft_score") <= 1) 
    & (pl.col("market_maker_score") <= 1)
    & (pl.col("institutional_score") <= 1)
    & (pl.col("retail_score") <= 1)
).collect()

original_order_id,venue,insertion_date,removal_date,latest_event_timestamp,number_of_updates,price_at_insertion,size_at_insertion,old_price_at_removal,old_size_at_removal,execution_size,insertion_level,best_bid_price_at_insertion,best_ask_price_at_insertion,lob_updates,associated_trade_ids,market_state_at_removal,size_diff_tally,order_lifetime,midpoint_at_insertion,abs_distance_to_midpoint,rel_distance_to_midpoint,log_order_lifetime,log_order_lifetime_bins,removal_mechanism,hft_score,market_maker_score,institutional_score,retail_score
i64,str,datetime[ns],datetime[ns],datetime[ns],u32,f64,i64,f64,i64,i64,i64,f64,f64,list[struct[3]],list[struct[1]],str,i64,i64,f64,f64,f64,f64,str,str,i32,i32,i32,i32
1693546881415785462,"""XETR""",2023-09-01 07:00:23.819527088,2023-09-01 11:00:00.096401017,2023-09-01 11:00:00.096401017,2,7.38,750,7.386,1000,0,56,7.114,7.12,"[{2023-09-01 10:21:06.085534951,7.44,1000}, {2023-09-01 10:21:58.330617502,7.386,1000}]",[],"""INTRADAY_AUCTION""",1000,14376276873,7.117,0.263,0.036954,23.388845,"""22-24""","""Cancel""",1,1,1,0
1693551474982396372,"""XETR""",2023-09-01 07:00:23.819462822,2023-09-01 07:19:02.139346568,2023-09-01 07:19:02.139346568,1,7.07,750,7.052,750,0,9,7.114,7.12,"[{2023-09-01 07:09:51.695592272,7.052,750}]",[],"""CONTINUOUS_TRADING""",750,1118319883,7.117,0.047,0.006604,20.835093,"""20-22""","""Cancel""",1,1,1,0
1693551623836068141,"""XETR""",2023-09-01 07:00:23.836076913,2023-09-01 07:59:53.952689433,2023-09-01 07:59:53.952689433,41,7.046,709,7.202,694,0,20,7.11,7.12,"[{2023-09-01 07:00:23.844735490,7.022,712}, {2023-09-01 07:00:23.845980223,7.048,709}, … {2023-09-01 07:58:18.410540402,7.202,694}]",[],"""CONTINUOUS_TRADING""",694,3570116612,7.115,0.069,0.009698,21.995864,"""20-22""","""Cancel""",1,1,1,0
1693551623836113288,"""XETR""",2023-09-01 07:00:23.836122368,2023-09-01 07:59:53.952704330,2023-09-01 07:59:53.952704330,51,7.204,694,7.334,681,0,24,7.11,7.12,"[{2023-09-01 07:00:23.843139442,7.192,695}, {2023-09-01 07:00:23.844159385,7.176,696}, … {2023-09-01 07:58:18.413532212,7.334,681}]",[],"""CONTINUOUS_TRADING""",681,3570116581,7.115,0.089,0.012509,21.995864,"""20-22""","""Cancel""",1,1,1,0
1693554015522135094,"""XETR""",2023-09-01 07:40:15.522145129,2023-09-01 07:46:42.699727677,2023-09-01 07:46:42.699727677,1,7.278,700,7.278,631,700,13,7.248,7.254,"[{2023-09-01 07:44:52.462064096,7.278,631}]","[{1693554292462036745}, {1693554402699695586}]","""CONTINUOUS_TRADING""",700,387177582,7.251,0.027,0.003724,19.774394,"""18-20""","""Trade (full)""",1,1,1,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1693575944520209336,"""XETR""",2023-09-01 13:45:44.520219722,2023-09-01 14:00:05.082313144,2023-09-01 14:00:05.082313144,5,7.364,678,7.32,683,0,24,7.424,7.43,"[{2023-09-01 13:47:15.575222927,7.35,680}, {2023-09-01 13:50:37.137121031,7.336,681}, … {2023-09-01 13:59:45.336378930,7.32,683}]",[],"""CONTINUOUS_TRADING""",683,860562093,7.427,0.063,0.008483,20.573096,"""20-22""","""Cancel""",1,1,1,0
1693575944520259736,"""XETR""",2023-09-01 13:45:44.520270132,2023-09-01 14:00:05.082333324,2023-09-01 14:00:05.082333324,6,7.492,667,7.444,671,0,29,7.424,7.43,"[{2023-09-01 13:47:04.441011067,7.478,668}, {2023-09-01 13:50:33.191284193,7.466,669}, … {2023-09-01 13:57:19.241147456,7.444,671}]",[],"""CONTINUOUS_TRADING""",671,860562063,7.427,0.065,0.008752,20.573096,"""20-22""","""Cancel""",1,1,1,0
1693576861211361192,"""XETR""",2023-09-01 14:01:01.211370228,2023-09-01 15:30:00.653300330,2023-09-01 15:30:00.653300330,12,7.288,686,7.302,684,0,22,7.348,7.364,"[{2023-09-01 14:01:19.047709457,7.3,684}, {2023-09-01 14:01:58.252266665,7.312,683}, … {2023-09-01 15:29:24.193501535,7.302,684}]",[],"""CLOSING_AUCTION""",684,5339441930,7.356,0.068,0.009244,22.398387,"""22-24""","""Cancel""",1,1,1,0


### Testing area


In [ ]:
quotes_inc_eu_test_filter = pl.col("trade_id").is_in([42943562011806, 118])
quotes_inc_eu \
    .filter(quotes_inc_eu_test_filter).sort(by=[pl.col("original_order_id"), pl.col("event_timestamp")]).collect()


In [ ]:
quotes_collapsed.filter(pl.col("removal_mechanism")=="Trade (partial)").collect().sample(10)

In [ ]:
test_ids = [50137] # [1081350376584932672, 1428746]
test_filter = pl.col("original_order_id").is_in(test_ids)

quotes_inc_eu \
    .filter(test_filter) \
    .sort(by=[pl.col("original_order_id"), pl.col("event_timestamp")]) \
    .collect() \
    .show(limit=15)

quotes_collapsed \
    .filter((pl.col("best_bid_price_at_insertion").is_null()) | (pl.col("best_ask_price_at_insertion").is_null())) \
    .collect() \
    # .show(limit=10)

# test_filter3 = pl.col("size_at_insertion") < pl.col("execution_size") # pl.col("size_before_removal").is_between(pl.col("size_at_insertion"), pl.col("execution_size"), closed="none")
# quotes_collapsed \
#     .filter(test_filter3) \
#     .collect() \
#     .show(limit=10)

In [ ]:
show_histogram(
    df=quotes_collapsed, 
    column_name="removal_mechanism",
    x_label="Removal mechanism"
)